# Section 4.3: integrality and Table 4
For **Recurrence formulas for Taylor expansions at CM points on genus-one and genus-two Shimura curves**, by Jingxuan Geng.

Run all cells with a **SageMath** kernel. This notebook is self-contained and uses exact rational arithmetic. It verifies every entry of Table 4 and the hypotheses of Lemma 4.1 for all twelve forms, including the **unit** condition on $Q(0)$. The conclusion is conditional on the recurrence data and initial coefficients in Table 2 and Appendix A (indices in Table 6, values in Table 7); those are the inputs here.

The normalization is $A(q)=\sum c_nq^n/n!$, $c_0=c_1=1$. We derive
$$F(v)=\left.\frac{L\mathcal R'(u)}{\phi'(u)}\right|_{u=\phi^{-1}(v)},\qquad
G(v)=\left.\frac L2(2\phi''(u)H(u)+\phi'(u)H'(u))\right|_{u=\phi^{-1}(v)}.$$
The independently transcribed Table 4 expressions below are comparison targets, not used to derive $F$ or $G$.

In [1]:
U.<u> = PolynomialRing(QQ)
K = U.fraction_field()
V.<v> = PolynomialRing(QQ)
# Each row: L, R(u), H(u), u0, u1, c2, c3, claimed minimal S.
data = {
 '14a': (-11/9, 3*(4-u)/8, 4*u*(-u^2+13*u-128),
         16, -352/3, -4, 214/9, 3),
 '15a': (-7/64, -4-4*u/3, 4*u*(-3*u^2-82*u-27),
         9, -63, -1/4, 27/16, 2),
 '21a': (-43/3969, -16*(17*u^2+392*u-2401)/(u-49)^2, 4*u*(-7*u^2+94*u-343),
         441/25, -9632/375, -905/3969, -1318/27783, 21),
 '33a': (-67/529, -4*(u^3+49*u^2+19*u+27)/(u*(u-1)^2), 4*u*(-3*u^2-10*u-243),
         9/25, -19296/2875, -72337/529, 9202302/12167, 23),
 '34a': (-11/25, (279*u^2+3104*u-39264)/(2*(15*u-16)^2), (u^2+4)*(-3*u^2+26*u-59),
         -15/4, -3179/80, -12/25, 2762/125, 5),
 '46a': (-844992/61701025, -(3*u^3+52*u^2+13824*u-311296)/(8*(u-64)^2), 4*u*(-u^2+45*u-512),
         3136/81, -15188992/636255, 27824049/123402050, -114816966372/484661551375, 15710),
 '26a': (-11/1764, 9/2-9/u-507/(2*u^2), 4*(-2*u^3+19*u^2-24*u-169),
         1, 44/21, -16/147, 274/3087, 21),
 '26b': (-11/4, 3+14/u-3/u^2, 4*(-169*u^3-24*u^2+19*u-2),
         1, 44, 40, -734, 1),
 '38a': (-11/324, -6*u-45/2-57/u-57/(2*u^2), 4*(-16*u^3-59*u^2-82*u-19),
         1, 44/9, -64/27, -62/27, 3),
 '38b': (-11/676, -16-52/u-30/u^2, 4*(-19*u^3-82*u^2-59*u-16),
         1, 44/13, -16/169, -878/2197, 13),
 '58a': (121/400, 5-144358/(243*u)-2523/u^2-229376/(243*(u+27))+65536/(9*(u+27)^2),
         4*(-2*u^3-78*u^2-862*u-1682), -11, -176/5, -4/5, 314/125, 5),
 '58b': (-11/144, -39625/243-34/u-3/u^2-163840/(243*(27*u+1))-65536/(243*(27*u+1)^2),
         4*(-1682*u^3-862*u^2-78*u-2), -1/11, -16/33, 140/9, 262/3, 3)
}

## The printed Table 4
Rows are $(\phi(u),v_1,P(v),Q(v),G(v))$. Factored denominators are retained so that their value at $v=0$ is visible.

In [2]:
printed = {
 '14a': ((u-16)/8, -44/3, 11/3, 1, 44/9*(12*v^2+35*v+30)),
 '15a': ((u-9)/3, -21, 7/16, 1, 7/32*(27*v^2+326*v+744)),
 '21a': ((441-25*u)/(16*u), 430/189, -43/147*(2*v+11), (1+v)^3,
         86/11907*(112*v^2-214*v-29)),
 '33a': ((9-25*u)/(16*u), 670/23, 201/529*(64*v^3+192*v^2+66*v-143), (1+v)^3,
         134/529*(1296*v^2+4070*v+3197)),
 '34a': (5*(4*u+15)/(11*(15*u-16)), 1, 3872/625*(33*v-5), 1,
         (287496*v^3-123420*v^2+19525*v-2125)/625),
 '46a': (81*u-3136, -15188992/7855,
         3912/61701025*(v^3-6144*v^2-112828416*v-309003812864), (v-2048)^3,
         62592/61701025*(v^2+3842*v+3333760)),
 '26a': ((u-1)/4, 11/21, -44/147*(3*v+43), (1+4*v)^3, 11/441*(12*v^2-13*v-1)),
 '26b': ((u-1)/4, 11, 88*(7*v+1), (1+4*v)^3, 11*(1014*v^2+531*v+67)),
 '38a': ((u-1)/4, 11/9, 44/27*(32*v^3+24*v^2-13*v-9), (1+4*v)^3,
         11/81*(96*v^2+107*v+31)),
 '38b': ((u-1)/4, 11/13, -176/169*(13*v+7), (1+4*v)^3, 11/169*(114*v^2+139*v+35)),
 '58a': ((u+11)/176, -1/5, -32/25*(11258929*v^4-149072*v^3+167706*v^2+308*v-52),
         (1+11*v)^3*(1-16*v)^3, -11/25*(1452*v^2+33*v-1)),
 '58b': ((11*u+1)/16, -1/3, -42592/9*v*(43*v-2)*(1227*v^2-86*v+2),
         (1-16*v)^3*(1-27*v)^3, (10092*v^2+1109*v-35)/9)
}

## Exact substitution and localization checks
A rational number lies in $\mathbb Z[1/S]$ precisely when every prime in its denominator divides $S$. It is a unit precisely when it is nonzero and its inverse also satisfies this test. Minimality is certified by the union of the prime divisors of the reduced denominators of $c_2,c_3$: every such prime must be inverted, while the differential system proves that no others are needed.

In [3]:
def substitute(r, x):
    r = K(r)
    return r.numerator()(x)/r.denominator()(x)

def primes(n):
    return set(ZZ(n).prime_divisors())

# Variables for the polynomial differential system: (A,A',A'',v,v',A^-1,Q(v)^-1).
T.<A,B,C,z,w,a,b> = PolynomialRing(QQ)
summary = []
assert set(data) == set(printed) and len(data) == 12
for name, (L,R,H,u0,u1,c2,c3,S) in data.items():
    phi,v1,P,Q,G = printed[name]
    phi,R = K(phi),K(R)
    P,Q,G = map(V, (P,Q,G))
    num,den = phi.numerator(),phi.denominator()
    assert max(num.degree(),den.degree()) <= 1
    assert num[1]*den[0]-num[0]*den[1] != 0
    psi = (den[0]*v-num[0])/(num[1]-den[1]*v)  # inverse Mobius map
    assert substitute(phi,psi) == v and substitute(phi,u0) == 0
    assert substitute(phi.derivative(),u0)*u1 == v1
    F = substitute(L*R.derivative()/phi.derivative(),psi)
    G_derived = substitute(L/2*(2*phi.derivative(2)*H+phi.derivative()*H.derivative()),psi)
    assert F == P/Q and G_derived == G, name
    assert u1^2 == L*H(u0)  # the first integral at q=0
    assert c2 == 3/2-L*substitute(R,u0)
    assert c3 == 6*c2-6-F(0)*v1  # the differentiated recurrence at q=0
    allowed = primes(S)
    integral = lambda x: primes(QQ(x).denominator()) <= allowed
    unit = lambda x: x != 0 and integral(x) and integral(1/QQ(x))
    assert unit(1) and unit(Q(0))
    assert all(integral(x) for x in [1,c2,v1]+P.list()+Q.list()+G.list())
    rhs = [B,C,6*B*C*a-6*B^3*a^2-P(z)*b*w*A^3,
           w,B*a*w+G(z)*A^2,-B*a^2,-Q.derivative()(z)*w*b^2]
    initial = [1,1,c2,0,v1,1,1/Q(0)]
    assert all(integral(x) for p in rhs for x in T(p).coefficients())
    assert all(integral(x) for x in initial)
    forced = primes(QQ(c2).denominator()) | primes(QQ(c3).denominator())
    assert forced == allowed and prod(forced) == S, name
    summary.append([name,S,v1,Q(0),sorted(forced),'PASS'])
show(table(summary,header_row=['form','minimal S','v1','Q(0)','necessary primes','Table 4 + lemma']))
print('All 12 rows verified exactly, including Q(0) units and minimal S.')

form,minimal S,v1,Q(0),necessary primes,Table 4 + lemma
14a,,,,,PASS
15a,,,,,PASS
21a,,,,,PASS
33a,,,,,PASS
34a,,,,,PASS
46a,,,,,PASS
26a,,,,,PASS
26b,,,,,PASS
38a,,,,,PASS
38b,,,,,PASS


All 12 rows verified exactly, including Q(0) units and minimal S.


## Why this certifies all coefficients
Set $B=A'$, $C=A''$, $w=v'$, $a=A^{-1}$ and $b=Q(v)^{-1}$. The seven components checked above satisfy
$$ (A,B,C,v,w,a,b)'=
\bigl(B,C,6BCa-6B^3a^2-P(v)bwA^3,w,Baw+G(v)A^2,-Ba^2,-Q'(v)wb^2\bigr). $$
Every coefficient of this polynomial vector field and every initial value belongs to $\mathbb Z[1/S]$. Differentiating any polynomial in these seven variables applies the polynomial derivation $\sum_i\mathrm{rhs}_i\partial_i$, which preserves this ring. Its repeated application to $A$ and $v$, followed by substitution of the initial values, therefore proves $A^{(n)}(0)=c_n$ and $v^{(n)}(0)=v_n$ lie in $\mathbb Z[1/S]$ **for every $n$**, not merely for a tested finite range. The inverses are valid locally because the checked constant terms are units.

This is the argument of Lemma 4.1 with every Table 4 specialization checked. The denominator witnesses show that the displayed positive squarefree $S$ is minimal. The case $26b$ has $S=1$, so all its normalized exponential coefficients are integers.